# Tuesday - Brute Force: CICIDS2017 initial cleaning

Recreated notebook. This is a clean, runnable equivalent of the previous analysis, **not an exact copy of the original notebook**. It does not change raw CSV files. It will write `ml/data/processed/Tuesday-WorkingHours-Cleaned.csv`; back up any existing output before rerunning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Run from a VS Code notebook anywhere inside the repository.
# If the repository cannot be found, set PROJECT_ROOT manually to its folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((p for p in candidates if (p / 'ml' / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find ml/data. Set PROJECT_ROOT to the repository folder.')
RAW_DIR = PROJECT_ROOT / "ml" / "data" / "raw" / "MachineLearningCVE"
PROCESSED_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Raw data:', RAW_DIR)

In [ ]:
RAW_FILE = RAW_DIR / 'Tuesday-WorkingHours.pcap_ISCX.csv'
OUTPUT_FILE = PROCESSED_DIR / 'Tuesday-WorkingHours-Cleaned.csv'
if not RAW_FILE.is_file():
    raise FileNotFoundError(f'Missing original CSV: {RAW_FILE}')
df = pd.read_csv(RAW_FILE, low_memory=False)
df.columns = df.columns.str.strip()
assert 'Label' in df.columns and 'Flow Duration' in df.columns
print('Original shape:', df.shape)
print(df['Label'].value_counts(dropna=False))
display(df.head())

## Data cleaning

Remove incomplete or infinite measurements, then exact duplicate records and negative flow durations. We preserve TCP window `-1` sentinel values rather than indiscriminately removing negative measurements.

In [ ]:
LABEL_MAPPING = {
    'Botnet': 'Bot', 'BOT': 'Bot', 'bot': 'Bot',
    'Infilteration': 'Infiltration', 'INFILTRATION': 'Infiltration', 'infiltration': 'Infiltration',
    'PORTSCAN': 'PortScan', 'Portscan': 'PortScan', 'portscan': 'PortScan', 'Port Scan': 'PortScan',
    'DDos': 'DDoS', 'Ddos': 'DDoS', 'ddos': 'DDoS', 'DDOS': 'DDoS',
    'Web Attack \ufffd Brute Force': 'Web Attack - Brute Force',
    'Web Attack \ufffd XSS': 'Web Attack - XSS',
    'Web Attack \ufffd Sql Injection': 'Web Attack - SQL Injection',
    'Web Attack - Sql Injection': 'Web Attack - SQL Injection',
}
# Repair actual Unicode replacement characters in corrupted source labels.
LABEL_MAPPING = {k.replace('\ufffd', '\ufffd'): v for k, v in LABEL_MAPPING.items()}

In [ ]:
clean_df = df.copy()
rows_original = len(clean_df)
clean_df['Label'] = clean_df['Label'].astype('string').str.strip().replace(LABEL_MAPPING)
clean_df.replace([np.inf, -np.inf], np.nan, inplace=True)
missing_cells = int(clean_df.isna().sum().sum())
missing_rows = int(clean_df.isna().any(axis=1).sum())
clean_df.dropna(inplace=True)
duplicates = int(clean_df.duplicated().sum())
clean_df.drop_duplicates(inplace=True)
invalid_duration = clean_df['Flow Duration'] < 0
negative_durations = int(invalid_duration.sum())
print('Invalid duration labels:')
print(clean_df.loc[invalid_duration, 'Label'].value_counts())
clean_df = clean_df.loc[~invalid_duration].reset_index(drop=True)
print('Original rows:', rows_original)
print('Missing rows:', missing_rows, '(missing/infinite cells:', missing_cells, ')')
print('Exact duplicates removed:', duplicates)
print('Negative flow durations removed:', negative_durations)
print('Final cleaned rows:', len(clean_df))
print('Total removed:', rows_original - len(clean_df))

## Traffic class distribution

In [ ]:
label_counts = clean_df['Label'].value_counts()
label_summary = pd.DataFrame({'Count': label_counts, 'Percentage': (label_counts / len(clean_df) * 100).round(4)})
display(label_summary)
ax = label_counts.plot(kind='bar', figsize=(11, 4), title='Cleaned traffic labels')
ax.set_ylabel('Records')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Feature inspection and anomalies

In [ ]:
features = clean_df.drop(columns=['Label'])
numeric = features.select_dtypes(include=[np.number])
print('Input features:', features.shape[1], '| Numeric features:', numeric.shape[1])
print('Non-numeric features:', features.columns.difference(numeric.columns).tolist())
constant = numeric.columns[numeric.nunique() <= 1].tolist()
print('Constant features:', len(constant), constant)
negative_count = (numeric < 0).sum()
display(negative_count[negative_count > 0].sort_values(ascending=False).rename('Negative count').to_frame())
zero_pct = ((numeric == 0).mean() * 100).sort_values(ascending=False)
display(zero_pct.head(25).rename('Zero %').to_frame())
display(numeric.describe().T)

## Final validation and export

In [ ]:
missing_final = int(clean_df.isna().sum().sum())
infinite_final = int(np.isinf(numeric.to_numpy()).sum())
duplicates_final = int(clean_df.duplicated().sum())
negative_duration_final = int((clean_df['Flow Duration'] < 0).sum())
print('Missing:', missing_final, 'Infinite:', infinite_final, 'Duplicates:', duplicates_final, 'Negative durations:', negative_duration_final)
assert missing_final == infinite_final == duplicates_final == negative_duration_final == 0
assert clean_df['Label'].notna().all()
clean_df.to_csv(OUTPUT_FILE, index=False)
print('Saved:', OUTPUT_FILE)
print('Rows:', len(clean_df), 'Columns:', len(clean_df.columns))
print('Final labels:')
print(clean_df['Label'].value_counts())
print('Validation passed!')